## **Nested JSON Data Transformation, Data Cleaning & Dimensional Modeling with PySpark**

### **Notebook objective**

> This notebook demonstrates an end-to-end data engineering workflow for transforming a highly nested JSON/JSONL dataset into structured tabular data, validating relationships and data quality, cleaning and normalizing the extracted data, building a Gold-layer analytical data model, and finally performing analytical exploration using Python.
---

In [1]:
# Importing Required Function
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

## **Load Existing Nested JSON Data**

### **Description**

> The dataset used in this project has already been generated in Python and stored in JSON and JSONL formats. In this stage, the existing datasets are loaded into PySpark DataFrames without modifying the source data. The purpose is to establish the raw starting point for the transformation pipeline while preserving the original nested structure.
---

In [2]:
def session_builder(session_name) :
  try :
    spark = SparkSession.builder\
            .appName(session_name)\
            .getOrCreate()

    print('=' * 50)
    print("Session Build Successfully.")
    print('=' * 50)
    return spark

  except Exception as e :
    raise RuntimeError(f"Faild To Create SparkSession : {e}") from e

def data_read(spark):

  dataset = {
      'orders_j' : '/content/orders.json',
      'orders_l' : '/content/orders.jsonl'
  }

  try :
    loaded_data = {
        name : spark.read.option("multiLine", True).option("inferSchema", "true").json(path)
        for name, path in dataset.items()
    }
    globals().update(loaded_data)
    print("All Provided Datasets Loaded Successfully.")
    print("=" * 50)

  except Exception as e :
    raise RuntimeError(f"Error Occured While Reading Data : {e}") from e

spark = session_builder("MySession")
data_read(spark)

Session Build Successfully.
All Provided Datasets Loaded Successfully.


In [ ]:
orders_j.show(1, truncate=False)
orders_l.show(1, truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## **Source Data and Schema Inspection**

### **Description**

> efore performing any transformation, the structure and contents of the nested datasets are inspected. This includes examining sample records, identifying nested structs and arrays, reviewing inferred data types, and understanding the hierarchical relationships between entities.

---

In [ ]:
print("=" * 50)
print("First Order Schema")
print("=" * 50)
orders_j.printSchema()

print("=" * 50)
print("Second Order Schema")
print("=" * 50)
orders_l.printSchema()

First Order Schema
root
 |-- audit_log: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- action: string (nullable = true)
 |    |    |-- actor: string (nullable = true)
 |    |    |-- carrier: string (nullable = true)
 |    |    |-- reason: string (nullable = true)
 |    |    |-- result: string (nullable = true)
 |    |    |-- ts: string (nullable = true)
 |-- customer: struct (nullable = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- loyalty: struct (nullable = true)
 |    |    |-- benefits: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- benefit_id: string (nullable = true)
 |    |    |    |    |-- name: string (nullable = true)
 |    |    |    |    |-- rules: struct (nullable = true)
 |    |    |    |    |    |-- categories: array (nullable = true)
 |    |    |    |    |    |    |-- element: string (containsNull = true)
 |    |    |    |    |    |-- min_spend: double (nu

## **Nested-to-Tabular Transformation**

### **Description**

> The highly nested JSON structures are transformed into tabular DataFrames using PySpark SQL. Nested structs are flattened into columns, while arrays are expanded using EXPLODE and EXPLODE_OUTER. Each transformation extracts a specific nested component while retaining the relevant parent identifiers and relationships.

In [3]:
orders_j.createOrReplaceTempView('orders_j')
orders_l.createOrReplaceTempView('orders_l')

In [ ]:
spark.sql(
    """
    SELECT
    -- extracting order audit_log info
      i.action::STRING   as action,
      i.actor::STRING    as actor,
      i.carrier::STRING  as carrier,
      i.reason::STRING   as reason,
      i.result::STRING   as result,
      i.ts::TIMESTAMP    as ts
    FROM orders_j        as o
    LATERAL VIEW EXPLODE(o.audit_log) audit_log as i
    """
).show(truncate=False)

+---------+--------+--------+----------------+------+-------------------+
|action   |actor   |carrier |reason          |result|ts                 |
+---------+--------+--------+----------------+------+-------------------+
|created  |system  |NULL    |NULL            |NULL  |2026-09-01 10:00:00|
|paid     |gateway |NULL    |NULL            |NULL  |2026-09-01 10:01:10|
|shipped  |wms     |BlueDart|NULL            |NULL  |2026-09-02 08:00:00|
|delivered|carrier |NULL    |NULL            |NULL  |2026-09-05 11:10:00|
|created  |system  |NULL    |NULL            |NULL  |2026-09-01 11:00:00|
|delivered|carrier |NULL    |NULL            |NULL  |2026-09-04 16:45:00|
|created  |system  |NULL    |NULL            |NULL  |2026-09-02 12:00:00|
|shipped  |wms     |NULL    |NULL            |NULL  |2026-09-03 10:00:00|
|created  |system  |NULL    |NULL            |NULL  |2026-09-03 08:00:00|
|cancelled|customer|NULL    |customer_request|NULL  |2026-09-03 09:30:00|
|created  |system  |NULL    |NULL     

In [ ]:
spark.sql(
   """
      SELECT
      -- extracting loyalty information
          customer.customer_id,
          customer.loyalty.points,
          customer.loyalty.tier,

      -- extracting benefits information
          b.benefit_id,
          b.name,

      -- extracting rules information
          b.rules.min_spend,
          c as categories,

      -- extracting profiles information
          o.customer.profile.age,
          o.customer.profile.name as profile_name,

      -- extracting contact information
          o.customer.profile.contact.phone,
          o.customer.profile.contact.email,

      -- extracting address information
          a.city,
          a.state,
          a.geo.latitude,
          a.geo.longitude,
          t as tags,

      -- extracting preferences information
          cc as communication_channels,
          fc as favorite_categories,
          pb as preferred_brands,

      -- extracting notification_settings information
          ou as order_updates,
          p  as promo,
          s  as security
      FROM orders_j as o
      LATERAL VIEW EXPLODE(o.customer.loyalty.benefits) benefit as b
      LATERAL VIEW EXPLODE(b.rules.categories) categorie as c
      LATERAL VIEW EXPLODE(o.customer.profile.contact.addresses) address as a
      LATERAL VIEW EXPLODE(a.tags) tag as t
      LATERAL VIEW EXPLODE(o.customer.profile.preferences.communication_channels) communication_channel as cc
      LATERAL VIEW EXPLODE(o.customer.profile.preferences.favorite_categories) favorite_categorie as fc
      LATERAL VIEW EXPLODE(o.customer.profile.preferences.preferred_brands) preferred_brand as pb
      LATERAL VIEW EXPLODE(o.customer.profile.preferences.notification_settings.order_updates) order_update as ou
      LATERAL VIEW EXPLODE(o.customer.profile.preferences.notification_settings.promo) pro as p
      LATERAL VIEW EXPLODE(o.customer.profile.preferences.notification_settings.security) sec as s
   """
).show(10, truncate=False)

+-----------+------+----+----------+------------+---------+-----------+---+-------------+----------------+-----------------------+------+-----+--------+---------+-------+----------------------+-------------------+----------------+-------------+-----+--------+
|customer_id|points|tier|benefit_id|name        |min_spend|categories |age|profile_name |phone           |email                  |city  |state|latitude|longitude|tags   |communication_channels|favorite_categories|preferred_brands|order_updates|promo|security|
+-----------+------+----+----------+------------+---------+-----------+---+-------------+----------------+-----------------------+------+-----+--------+---------+-------+----------------------+-------------------+----------------+-------------+-----+--------+
|C-2006     |7800  |Gold|B-EA      |Early Access|NULL     |Electronics|52 |Liam O'Connor|+353-85-555-0101|liam.oconnor@example.ie|Dublin|D    |53.3498 |-6.2603  |primary|email                 |Electronics        |ViewMax

In [ ]:
spark.sql(
    """
      SELECT
       --extracting expedite information
          o.fees.expedite.amount  as expedite_amount ,
          o.fees.expedite.taxable as expedite_taxable,

       --extracting expedite information
          o.fees.gift_wrap.amount  as gift_wrap_amount,
          o.fees.gift_wrap.taxable as gift_wrap_taxable,

       --extracting insurance information
          o.fees.insurance.amount  as insurance_amount,
          o.fees.insurance.taxable as insurance_taxable,

        --extracting shipping information
          o.fees.shipping.amount  as shipping_amount,
          o.fees.shipping.taxable as shipping_taxable
      FROM orders_j as o
    """
).show(10, truncate=False)

+---------------+----------------+----------------+-----------------+----------------+-----------------+---------------+----------------+
|expedite_amount|expedite_taxable|gift_wrap_amount|gift_wrap_taxable|insurance_amount|insurance_taxable|shipping_amount|shipping_taxable|
+---------------+----------------+----------------+-----------------+----------------+-----------------+---------------+----------------+
|NULL           |NULL            |4.5             |true             |NULL            |NULL             |0.0            |false           |
|NULL           |NULL            |NULL            |NULL             |NULL            |NULL             |3.99           |false           |
|NULL           |NULL            |NULL            |NULL             |NULL            |NULL             |0.0            |false           |
|NULL           |NULL            |NULL            |NULL             |NULL            |NULL             |NULL           |NULL            |
|NULL           |NULL            |

In [ ]:
spark.sql(
    """
      SELECT
        -- extracting order_items information
            i.item_id,
            i.quantity,

        -- extracting discounts information
            d.amount,
            d.code,
            d.discount_type,

        -- extracting discounts_cinditions information
            c.condition_type,
            c.value,

        -- extracting pricing information
            i.pricing.discount,
            i.pricing.tax,
            i.pricing.unit_price,

        -- extracting product information
            i.product.product_id,
            i.product.name as product_name,
            i.product.category as product_category,
            i.product.brand,

        -- extracting product_attributes information
            i.product.attributes.anc,
            i.product.attributes.bluetooth,
            i.product.attributes.color,
            i.product.attributes.dpi,
            i.product.attributes.fit,
            i.product.attributes.flavor,
            i.product.attributes.format,
            i.product.attributes.gender,
            i.product.attributes.gps,
            i.product.attributes.language,
            i.product.attributes.material,
            i.product.attributes.panel,
            i.product.attributes.pressure,
            i.product.attributes.ram,
            i.product.attributes.refresh,
            i.product.attributes.scent,
            i.product.attributes.storage,
            i.product.attributes.strap,
            i.product.attributes.thickness,
            i.product.attributes.vegan,
            i.product.attributes.wireless,

        -- extracting supplier and supplier_certification information
            i.product.supplier.name as supplier_name,
            ct.code,
            ct.expires,

        -- extracting product_variants information
            v.variant_id,
            v.name as variant_name,
            v.attributes.accessory,
            v.attributes.color,
            v.attributes.flavor,
            v.attributes.format,
            v.attributes.language,
            v.attributes.layout,
            v.attributes.plug,
            v.attributes.ram,
            v.attributes.scent,
            v.attributes.size,
            v.attributes.strap,
            v.attributes.thickness,
            v.attributes.volume

      FROM orders_j as o
      LATERAL VIEW EXPLODE_OUTER(o.items) item as i
      LATERAL VIEW EXPLODE_OUTER(i.discounts) discount as d
      LATERAL VIEW EXPLODE_OUTER(d.conditions) condition as c
      LATERAL VIEW EXPLODE_OUTER(i.product.supplier.certifications) certification as ct
      LATERAL VIEW EXPLODE_OUTER(i.variants) variant as v
    """
).show(30, truncate=False)

+-----------+--------+------+-----------+-------------+--------------+--------------+--------+-----+----------+----------+-----------------------+----------------+----------+----+---------+------+----+----+---------+------+------+----+--------+--------+-----+--------+----+-------+-----+-------+-----+---------+-----+--------+-------------------+-------+----------+------------+------------------+---------+------+------+------+--------+------+----+----+-----+----+-----+---------+------+
|item_id    |quantity|amount|code       |discount_type|condition_type|value         |discount|tax  |unit_price|product_id|product_name           |product_category|brand     |anc |bluetooth|color |dpi |fit |flavor   |format|gender|gps |language|material|panel|pressure|ram |refresh|scent|storage|strap|thickness|vegan|wireless|supplier_name      |code   |expires   |variant_id  |variant_name      |accessory|color |flavor|format|language|layout|plug|ram |scent|size|strap|thickness|volume|
+-----------+--------+

```
 |-- labels: struct (nullable = true)
 |    |-- backorder: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- fraud_flags: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- warehouse: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
```

In [ ]:
spark.sql(
    """
    SELECT
      b AS backorder_value,
      f AS fraud_flags,
      w AS warehouse
    FROM orders_j as o
    LATERAL VIEW EXPLODE_OUTER(o.labels.backorder) backorder as b
    LATERAL VIEW EXPLODE_OUTER(o.labels.fraud_flags) fraud_flags as f
    LATERAL VIEW EXPLODE_OUTER(o.labels.warehouse) warehouse as w
    """
).show(20,truncate=False)

+---------------+----------------+---------+
|backorder_value|fraud_flags     |warehouse|
+---------------+----------------+---------+
|NULL           |NULL            |WH-WEST  |
|NULL           |NULL            |NULL     |
|NULL           |NULL            |WH-SOUTH |
|NULL           |NULL            |NULL     |
|NULL           |address_mismatch|NULL     |
|NULL           |high_value      |WH-EU1   |
|NULL           |high_value      |WH-EU2   |
|NULL           |NULL            |NULL     |
|NULL           |NULL            |WH-CN1   |
|NULL           |NULL            |WH-GCC   |
|NULL           |NULL            |WH-US-W  |
|NULL           |NULL            |WH-US-W2 |
|NULL           |NULL            |WH-ES1   |
|NULL           |NULL            |NULL     |
|NULL           |NULL            |WH-WEST  |
|NULL           |NULL            |WH-JP1   |
|P114           |NULL            |WH-SE1   |
|NULL           |NULL            |WH-SOUTH |
|NULL           |NULL            |WH-SOUTH |
|NULL     

```
 |-- metadata: struct (nullable = true)
 |    |-- campaign: string (nullable = true)
 |    |-- cancel_reason: string (nullable = true)
 |    |-- device: string (nullable = true)
 |    |-- gift: string (nullable = true)
 |    |-- ingest_batch: string (nullable = true)
 |    |-- return_reason: string (nullable = true)
 |    |-- source: string (nullable = true)
```

In [ ]:
spark.sql(
    """
    SELECT
      o.metadata.campaign,
      o.metadata.cancel_reason,
      o.metadata.device,
      o.metadata.gift,
      o.metadata.ingest_batch,
      o.metadata.return_reason,
      o.metadata.source
    FROM orders_j as o
    """
).show(10, truncate=False)

+---------+-------------+-------+----+------------+-------------+------+
|campaign |cancel_reason|device |gift|ingest_batch|return_reason|source|
+---------+-------------+-------+----+------------+-------------+------+
|FALL26   |NULL         |ios    |NULL|NULL        |NULL         |web   |
|NULL     |NULL         |android|NULL|NULL        |NULL         |app   |
|HOME-FEST|NULL         |NULL   |true|NULL        |NULL         |web   |
|NULL     |NULL         |NULL   |NULL|NULL        |NULL         |NULL  |
|NULL     |changed_mind |NULL   |NULL|NULL        |NULL         |web   |
|NULL     |NULL         |desktop|NULL|NULL        |NULL         |web   |
|NULL     |NULL         |android|NULL|B-0904-1    |NULL         |app   |
|NULL     |NULL         |ios    |NULL|NULL        |NULL         |app   |
|NULL     |NULL         |desktop|NULL|NULL        |NULL         |web   |
|LABOR-DAY|NULL         |desktop|NULL|NULL        |NULL         |web   |
+---------+-------------+-------+----+------------+

```
 |-- order_date: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
```

In [ ]:
spark.sql(
    """
    SELECT
    order_date,
    order_id,
    order_status
    FROM orders_j as o
    """
).show(10, truncate=False)

+----------+--------+------------+
|order_date|order_id|order_status|
+----------+--------+------------+
|2026-09-01|ORD-1001|DELIVERED   |
|2026-09-01|ORD-1002|delivered   |
|2026-09-02|ORD-1003|Shipped     |
|2026-09-02|ORD-1004|Pending     |
|2026-09-03|ORD-1005|CANCELLED   |
|2026-09-03|ORD-1006|SHIPPED     |
|2026-09-04|ORD-1007|Pending     |
|2026-09-05|ORD-1008|delivered   |
|2026-09-05|ORD-1009|Shipped     |
|2026-09-06|ORD-1010|DELIVERED   |
+----------+--------+------------+
only showing top 10 rows


```
 |-- payments: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- attempts: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- attempt_no: long (nullable = true)
 |    |    |    |    |-- gateway_response: struct (nullable = true)
 |    |    |    |    |    |-- avs: string (nullable = true)
 |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |-- result: string (nullable = true)
 |    |    |-- metadata: struct (nullable = true)
 |    |    |    |-- 3ds: string (nullable = true)
 |    |    |    |-- account: string (nullable = true)
 |    |    |    |-- card_no: string (nullable = true)
 |    |    |    |-- country: string (nullable = true)
 |    |    |    |-- iban_last4: string (nullable = true)
 |    |    |    |-- issuer: string (nullable = true)
 |    |    |    |-- last4: string (nullable = true)
 |    |    |    |-- mandate: string (nullable = true)
 |    |    |    |-- network: string (nullable = true)
 |    |    |    |-- payer_id: string (nullable = true)
 |    |    |    |-- refund_id: string (nullable = true)
 |    |    |    |-- vpa: string (nullable = true)
 |    |    |    |-- wallet: string (nullable = true)
 |    |    |-- method: string (nullable = true)
 |    |    |-- payment_id: string (nullable = true)
 |    |    |-- status: string (nullable = true)
```

In [ ]:
spark.sql(
    """
    SELECT
    -- extracting the payment information
      pay.payment_id,
      pay.amount,
      pay.method,
      pay.status,

  -- extracting the payment_attempt information
      at.attempt_no,
      at.result,
      at.gateway_response.avs,
      at.gateway_response.code,

  -- extracting the metadata information
      pay.metadata.3ds,
      pay.metadata.account,
      pay.metadata.card_no,
      pay.metadata.country,
      pay.metadata.iban_last4,
      pay.metadata.issuer,
      pay.metadata.last4,
      pay.metadata.mandate,
      pay.metadata.network,
      pay.metadata.payer_id,
      pay.metadata.refund_id,
      pay.metadata.vpa,
      pay.metadata.wallet
    FROM orders_j as o
    LATERAL VIEW EXPLODE(o.payments) payment as pay
    LATERAL VIEW EXPLODE(pay.attempts) attempt as at
    """
).show(30, truncate=False)

+----------+-------+------+--------+----------+--------+----+----+----+----------------+-------+-------+----------+------+-----+-------+----------+--------+---------+------------+---------+
|payment_id|amount |method|status  |attempt_no|result  |avs |code|3ds |account         |card_no|country|iban_last4|issuer|last4|mandate|network   |payer_id|refund_id|vpa         |wallet   |
+----------+-------+------+--------+----------+--------+----+----+----+----------------+-------+-------+----------+------+-----+-------+----------+--------+---------+------------+---------+
|PAY-1001-1|1545.77|card  |captured|1         |approved|Y   |00  |true|NULL            |NULL   |NULL   |NULL      |NULL  |4242 |NULL   |visa      |NULL    |NULL     |NULL        |NULL     |
|PAY-1003-1|98.18  |card  |captured|1         |declined|NULL|05  |NULL|NULL            |NULL   |NULL   |NULL      |NULL  |0007 |NULL   |mastercard|NULL    |NULL     |NULL        |NULL     |
|PAY-1003-1|98.18  |card  |captured|2         |app

```
 |-- shipments: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- carrier: string (nullable = true)
 |    |    |-- events: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- location: struct (nullable = true)
 |    |    |    |    |    |-- city: string (nullable = true)
 |    |    |    |    |    |-- country: string (nullable = true)
 |    |    |    |    |-- status: string (nullable = true)
 |    |    |    |    |-- timestamp: string (nullable = true)
 |    |    |-- packages: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- contents: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- item_id: string (nullable = true)
 |    |    |    |    |    |    |-- qty: long (nullable = true)
 |    |    |    |    |-- dimensions: struct (nullable = true)
 |    |    |    |    |    |-- height: double (nullable = true)
 |    |    |    |    |    |-- length: double (nullable = true)
 |    |    |    |    |    |-- width: double (nullable = true)
 |    |    |    |    |-- package_id: string (nullable = true)
 |    |    |    |    |-- weight: double (nullable = true)
 |    |    |-- shipment_id: string (nullable = true)
 |    |    |-- tracking: string (nullable = true)
```

In [ ]:
spark.sql(
    """
    SELECT
    --extracting shipment information
      sp.shipment_id,
      sp.carrier,
      sp.tracking,

   -- extracting event information
      ev.location.country,
      ev.location.city,
      ev.status,
      ev.timestamp,

   -- extracting packages information
      pkg.package_id,
      pkg.weight,

  -- extracting shipment_packages_contents information
     cnt.item_id,
     cnt.qty as quantity,

  -- extracting shipment_packages_dimensions information
     pkg.dimensions.height,
     pkg.dimensions.length,
     pkg.dimensions.width

    FROM orders_j as o
    LATERAL VIEW EXPLODE(o.shipments) shipment as sp
    LATERAL VIEW EXPLODE(sp.events) event as ev
    LATERAL VIEW EXPLODE(sp.packages) package as pkg
    LATERAL VIEW EXPLODE(pkg.contents) content as cnt
    """
).show(30, truncate=False)

+-----------+----------+------------------+-------+---------+----------+--------------------+------------+------+-----------+--------+------+------+-----+
|shipment_id|carrier   |tracking          |country|city     |status    |timestamp           |package_id  |weight|item_id    |quantity|height|length|width|
+-----------+----------+------------------+-------+---------+----------+--------------------+------------+------+-----------+--------+------+------+-----+
|SHP-1001-1 |BlueDart  |BD778899          |IN     |Mumbai   |picked_up |2026-09-02T08:00:00Z|PKG-1001-1-1|2.4   |ORD-1001-I1|1       |8.0   |40.0  |30.0 |
|SHP-1001-1 |BlueDart  |BD778899          |IN     |Mumbai   |picked_up |2026-09-02T08:00:00Z|PKG-1001-1-2|0.9   |ORD-1001-I2|2       |10.0  |25.0  |20.0 |
|SHP-1001-1 |BlueDart  |BD778899          |IN     |Nagpur   |in_transit|2026-09-03T14:30:00Z|PKG-1001-1-1|2.4   |ORD-1001-I1|1       |8.0   |40.0  |30.0 |
|SHP-1001-1 |BlueDart  |BD778899          |IN     |Nagpur   |in_transi

In [ ]:
from pyspark import SparkConf , SparkContext

conf = SparkConf()\
       .setAppName("TheSparkApplication")\
       .setMaster("local[*]")

# Use getOrCreate to avoid ValueError when a SparkContext is already running
sc = SparkContext.getOrCreate(conf=conf)

In [ ]:
print(sc.version)
print(sc.pythonVer)
print(sc.master)

4.0.4
3.13
local[*]


In [ ]:
numb = range(1, 100)
spark_data = sc.parallelize(numb)